# Часть A. Подготовка

Перед запуском выберите GPU: Runtime → Change runtime type → L4, A100 или H100.

**1. Подключаем Google Drive.** Там хранится всё важное: машина Colab очищается после каждой сессии. Colab попросит разрешение — согласитесь.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

**2. Создаём папки проекта.** На Drive — для чекпоинтов и результатов, на диске машины — для моделей и кэша (их можно скачать заново).

In [ ]:
import os

DRIVE = "/content/drive/MyDrive/unlearning_data"  # папка проекта на Google Drive (постоянная)
FAST = "/content/fast"                             # папка на диске машины (очищается после сессии)

# exist_ok=True — если папка уже есть, ничего не делать
os.makedirs(DRIVE + "/saves", exist_ok=True)        # чекпоинты моделей
os.makedirs(DRIVE + "/results_raw", exist_ok=True)  # сырые результаты атак
os.makedirs(FAST + "/hf_home", exist_ok=True)       # кэш Hugging Face
os.makedirs(FAST + "/models", exist_ok=True)        # скачанные модели

**3. Задаём переменные окружения.** По ним программы находят папки из шага 2. Должны напечататься два пути.

In [ ]:
os.environ["BIG"] = DRIVE                       # «большой диск» из плана
os.environ["HF_HOME"] = FAST + "/hf_home"
os.environ["MODELS"] = FAST + "/models"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # меньше лишних предупреждений
os.environ["PYTHONUNBUFFERED"] = "1"            # вывод программ сразу попадает в лог

!echo $HF_HOME
!echo $MODELS

**4. Подключаем токен Hugging Face.** Должно напечататься ваше имя на Hugging Face. Токен добавляется заранее, один раз: huggingface.co → Settings → Access Tokens → создать токен типа Read; в Colab слева 🔑 Secrets → добавить `HF_TOKEN` и включить доступ для блокнота.

In [ ]:
from google.colab import userdata
from huggingface_hub import whoami

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Hugging Face:", whoami()["name"])

**5. Смотрим GPU.** В таблице должна быть L4, A100 или H100 (T4 для обучения не подходит), а справа вверху — CUDA Version 12 или выше.

In [ ]:
!nvidia-smi

**6. Записываем характеристики машины в журнал** (`journal.md` на Drive, пригодится для главы 3 диплома). Выполняйте один раз для каждой новой модели GPU.

In [ ]:
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

import psutil

# вывод команд nvidia-smi — в переменные
gpu = !nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader
cuda = !nvidia-smi | grep -o "CUDA Version: [0-9.]*"

ram = round(psutil.virtual_memory().total / 1e9)           # ОЗУ в гигабайтах
disk = round(shutil.disk_usage("/content").free / 1e9)     # свободное место на диске машины в гигабайтах
today = datetime.now(ZoneInfo("Europe/Moscow")).date()     # дата по Москве: часы Colab идут по UTC

text = f"""
## {today} — Проверка машины (Colab)
- GPU (имя, память, compute capability, драйвер): {gpu[0]}
- {cuda[0]}
- CPU: {os.cpu_count()} ядер, ОЗУ: {ram} ГБ, свободно на диске машины: {disk} ГБ
- Большой диск: {DRIVE}
"""

with open(DRIVE + "/journal.md", "a", encoding="utf-8") as f:
    f.write(text)

print(text)